# Lezione 3 · Come impara una rete

Lo scandaglio nella nebbia, il cesto a fine giornata, l'errore che risale filo per filo, e i conti del cantiere dei grandi.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

def softmax(z):
    e = np.exp(np.array(z, dtype=float) - np.max(z))
    return e / e.sum()

## I punti persi, cioè la loss

*Nel racconto: le tavolette del primo dan; «imparare vuol dire perdere sempre meno punti, su tutte le serate insieme».*

In [2]:
# dopo «il vento gira a», quanto paga il modello per «maestrale»
controlla("se gli dava 0,60", -math.log(0.60), 0.51)
controlla("se gli dava 0,05", -math.log(0.05), 3.0)

✓  se gli dava 0,60: calcolato 0.5108, nel libro 0.51
✓  se gli dava 0,05: calcolato 2.996, nel libro 3


## Il gradiente e la discesa

*Nel racconto: lo scandaglio nella nebbia, il timone che serpeggia, la barca che rimbalza da una sponda all'altra.*

Un fondale con un solo parametro, L(w) = (w − 3)²: la conca è a w = 3, la pendenza è 2(w − 3).

In [3]:
def discesa(eta, passi, w=0.0):
    strada = []
    for _ in range(passi):
        w = w - eta * 2 * (w - 3)
        strada.append(w)
    return strada

for nome, v, atteso in zip(["w₁", "w₂", "w₃"], discesa(0.1, 3), [0.6, 1.08, 1.46]):
    controlla(f"η = 0,1: {nome}", v, atteso)
controlla("η = 0,5: un solo passo, e sei nella conca", discesa(0.5, 1)[0], 3)
controlla("η = 0,01: distanza dalla conca dopo cento passi", 3 - discesa(0.01, 100)[-1], 0.40)
for nome, v, atteso in zip(["w₁", "w₂", "w₃"], discesa(1.1, 3), [6.6, -1.32, 8.18]):
    controlla(f"η = 1,1: {nome}", v, atteso)
# arriva alla conca solo se 0 < η < 1
assert abs(discesa(0.99, 2000)[-1] - 3) < 1e-6 and abs(discesa(1.01, 2000)[-1] - 3) > 1e6

✓  η = 0,1: w₁: calcolato 0.6, nel libro 0.6
✓  η = 0,1: w₂: calcolato 1.08, nel libro 1.08
✓  η = 0,1: w₃: calcolato 1.464, nel libro 1.46
✓  η = 0,5: un solo passo, e sei nella conca: calcolato 3, nel libro 3
✓  η = 0,01: distanza dalla conca dopo cento passi: calcolato 0.3979, nel libro 0.4
✓  η = 1,1: w₁: calcolato 6.6, nel libro 6.6
✓  η = 1,1: w₂: calcolato -1.32, nel libro -1.32
✓  η = 1,1: w₃: calcolato 8.184, nel libro 8.18


## Il mini-batch

*Nel racconto: tre o quattro calate con le onde che dondolano la barca.*

In [4]:
controlla("errore con B = 100 rispetto a B = 4 (un quinto)", math.sqrt(4) / math.sqrt(100), 1 / 5)
controlla("costo di un passo con B = 100 rispetto a B = 4", 100 / 4, 25)

# la prova con le onde: la media di B calate rumorose attorno alla pendenza vera
rng = np.random.default_rng(0)
calate = rng.normal(1.0, 2.0, size=(20000, 100))
print(f"   dispersione della stima: B = 4 → {calate[:, :4].mean(axis=1).std():.2f}, B = 100 → {calate.mean(axis=1).std():.2f}")

✓  errore con B = 100 rispetto a B = 4 (un quinto): calcolato 0.2, nel libro 0.2
✓  costo di un passo con B = 100 rispetto a B = 4: calcolato 25, nel libro 25
   dispersione della stima: B = 4 → 1.00, B = 100 → 0.20


## Il cesto a fine giornata

*Nel racconto: il pescatore e la sua ricompensa; «meglio del solito».*

Due baie con punteggi uguali; il pescatore va in A e pesca 8 kg contro i 5 abituali.

In [5]:
punteggi = np.array([0.0, 0.0])
p = softmax(punteggi)
R, solito, eta = 8, 5, 0.1
derivata = np.array([1 - p[0], -p[1]])       # di ln p(A) rispetto ai due punteggi
passo = eta * (R - solito) * derivata
controlla("il punteggio di A sale di", passo[0], 0.15)
controlla("il punteggio di B scende di", -passo[1], 0.15)
controlla("p(A) dopo una giornata buona", softmax(punteggi + passo)[0], 0.574, 0.001)
controlla("p(A) se avesse pescato 2 kg (%)", softmax(punteggi + eta * (2 - solito) * derivata)[0] * 100, 43, 0.5)

✓  il punteggio di A sale di: calcolato 0.15, nel libro 0.15
✓  il punteggio di B scende di: calcolato 0.15, nel libro 0.15
✓  p(A) dopo una giornata buona: calcolato 0.5744, nel libro 0.574
✓  p(A) se avesse pescato 2 kg (%): calcolato 42.56, nel libro 43


## La backpropagation, filo per filo

*Nel racconto: l'errore che risale in rosso sul foglio dei tre cerchi.*

La sera in cui non sei andato: la rete ha detto 0,85, la verità era 0. Come nel libro, p = 0,845 è arrotondato a 0,85, e il learning rate è 0,1.

In [6]:
W1 = np.array([[2.0, 0.0, 1.0], [-3.0, 2.0, 0.0]]); b1 = np.array([-1.0, 0.5])
w2 = np.array([1.5, -2.0]); b2 = -0.5
x = np.array([0.8, 1, 1]); y = 0; eta = 0.1

def avanti(W1, b1, w2, b2, x=x):
    z1 = W1 @ x + b1
    h = np.maximum(0, z1)
    z2 = float(w2 @ h) + b2
    return z1, h, z2, sigmoide(z2)

z1, h, z2, p = avanti(W1, b1, w2, b2)
controlla("la rete dice", p, 0.845, 0.001)

d_out = 0.85 - y                                  # primo tempo: risposta meno verità
g_w2 = d_out * h                                  # secondo tempo: i fili che arrivano alla decisione
controlla("colpa slancio → decisione", g_w2[0], 1.36)
controlla("colpa pioggia che pesa → decisione", g_w2[1], 0.085, 0.0005)
d1 = d_out * w2 * (z1 > 0)                        # terzo tempo: l'errore risale ai due cerchi
controlla("errore allo slancio", d1[0], 1.275, 0.0005)
controlla("errore alla pioggia che pesa", d1[1], -1.7)
g_W1 = np.outer(d1, x)                            # quarto tempo: i fili che partono dai sassolini
for riga, nome, attesi in [(0, "slancio", [1.02, 1.275, 1.275]), (1, "pioggia che pesa", [-1.36, -1.7, -1.7])]:
    for fatto, v, a in zip(["energia", "pioggia", "compagno"], g_W1[riga], attesi):
        controlla(f"colpa {fatto} → {nome}", v, a, 0.0005)

# l'ultimo gesto: la correzione di tutti gli undici parametri
W1n, b1n, w2n, b2n = W1 - eta * g_W1, b1 - eta * d1, w2 - eta * g_w2, b2 - eta * d_out
for nome, v, attesi in [("slancio", W1n[0], [1.898, -0.128, 0.873]), ("pioggia che pesa", W1n[1], [-2.864, 2.17, 0.17])]:
    for fatto, vv, a in zip(["energia", "pioggia", "compagno"], v, attesi):
        controlla(f"dopo: {fatto} → {nome}", vv, a, 0.0006)
controlla("dopo: bias dello slancio", b1n[0], -1.128, 0.0006)
controlla("dopo: bias della pioggia che pesa", b1n[1], 0.67)
controlla("dopo: slancio → decisione", w2n[0], 1.364, 0.0005)
controlla("dopo: pioggia che pesa → decisione", w2n[1], -2.009, 0.0006)
controlla("dopo: bias della decisione", b2n, -0.585, 0.0005)

_, hn, z2n, pn = avanti(W1n, b1n, w2n, b2n)
controlla("il nuovo slancio (nei conti 1,136; nel racconto circa 1,14)", hn[0], 1.136, 0.001)
controlla("la nuova pioggia che pesa (nei conti 0,719; nel racconto circa 0,72)", hn[1], 0.719, 0.001)
controlla("la nuova spinta", z2n, -0.48)
controlla("la vela, da 85% a (%)", pn * 100, 38, 0.5)

# in forma di matrice: la trasposta, per la pendenza della ReLU
assert np.allclose(d1, (w2 * d_out) * (z1 > 0))

✓  la rete dice: calcolato 0.8455, nel libro 0.845
✓  colpa slancio → decisione: calcolato 1.36, nel libro 1.36
✓  colpa pioggia che pesa → decisione: calcolato 0.085, nel libro 0.085
✓  errore allo slancio: calcolato 1.275, nel libro 1.275
✓  errore alla pioggia che pesa: calcolato -1.7, nel libro -1.7
✓  colpa energia → slancio: calcolato 1.02, nel libro 1.02
✓  colpa pioggia → slancio: calcolato 1.275, nel libro 1.275
✓  colpa compagno → slancio: calcolato 1.275, nel libro 1.275
✓  colpa energia → pioggia che pesa: calcolato -1.36, nel libro -1.36
✓  colpa pioggia → pioggia che pesa: calcolato -1.7, nel libro -1.7
✓  colpa compagno → pioggia che pesa: calcolato -1.7, nel libro -1.7
✓  dopo: energia → slancio: calcolato 1.898, nel libro 1.898
✓  dopo: pioggia → slancio: calcolato -0.1275, nel libro -0.128
✓  dopo: compagno → slancio: calcolato 0.8725, nel libro 0.873
✓  dopo: energia → pioggia che pesa: calcolato -2.864, nel libro -2.864
✓  dopo: pioggia → pioggia che pesa: calcolato

## Lo scandaglio a mano e l'ecoscandaglio elettronico

*Nel racconto: il piombo calato in una direzione alla volta, e l'ecoscandaglio elettronico che legge il fondale in un colpo solo.*

In [7]:
def loss(W1, b1, w2, b2):
    return -math.log(1 - avanti(W1, b1, w2, b2)[3])       # y = 0

L0 = loss(W1, b1, w2, b2)
controlla("la loss sulla serata", L0, 1.8678, 0.0001)
passo_h = 0.001
L1 = loss(W1, b1, w2 + np.array([passo_h, 0]), b2)
controlla("spostando slancio → decisione da 1,5 a 1,501", L1, 1.8691, 0.0001)
controlla("differenze finite", (L1 - L0) / passo_h, 1.353, 0.001)
controlla("la colpa trovata risalendo, 0,8455 × 1,6", p * h[0], 1.353, 0.001)
conta("passaggi in avanti con il piombo, per 11 parametri", 11 + 1, 12)

# il gradient check su tutti gli undici parametri, con la p non arrotondata
d = p - y
esatte = np.concatenate([np.outer(d * w2 * (z1 > 0), x).ravel(), d * w2 * (z1 > 0), d * h, [d]])
theta = np.concatenate([W1.ravel(), b1, w2, [b2]])
def loss_di(t):
    return loss(t[:6].reshape(2, 3), t[6:8], t[8:10], t[10])
finite = np.array([(loss_di(theta + 1e-6 * np.eye(11)[i]) - loss_di(theta)) / 1e-6 for i in range(11)])
assert np.allclose(esatte, finite, atol=1e-4)
print("   gradient check: le undici colpe risalite coincidono con le undici calate del piombo")

✓  la loss sulla serata: calcolato 1.868, nel libro 1.8678
✓  spostando slancio → decisione da 1,5 a 1,501: calcolato 1.869, nel libro 1.8691
✓  differenze finite: calcolato 1.353, nel libro 1.353
✓  la colpa trovata risalendo, 0,8455 × 1,6: calcolato 1.353, nel libro 1.353
✓  passaggi in avanti con il piombo, per 11 parametri: calcolato 12, nel libro 12
   gradient check: le undici colpe risalite coincidono con le undici calate del piombo


## Perché non si torna indietro invertendo

*Nel racconto: la navigazione che non si disfa; «−0,4 o −400».*

In [8]:
for h1, h2 in [(1.6, 0.1), (2.0, 0.4), (1.2, -0.2)]:
    controlla(f"1,5 × {h1} − 2,0 × {h2} − 0,5", 1.5 * h1 - 2.0 * h2 - 0.5, 1.7)
assert max(0, -0.4) == max(0, -400) == 0

✓  1,5 × 1.6 − 2,0 × 0.1 − 0,5: calcolato 1.7, nel libro 1.7
✓  1,5 × 2.0 − 2,0 × 0.4 − 0,5: calcolato 1.7, nel libro 1.7
✓  1,5 × 1.2 − 2,0 × -0.2 − 0,5: calcolato 1.7, nel libro 1.7


## I neuroni morti

*Nel racconto: lo slancio ucciso da un passo troppo lungo; la scogliera con una fessura.*

Il passo η = 2, venti volte quello del racconto.

In [9]:
slancio = W1[0] - 2 * g_W1[0]
bias = b1[0] - 2 * d1[0]
for nome, v, a in zip(["energia", "pioggia", "compagno"], slancio, [-0.04, -2.55, -1.55]):
    controlla(f"lo slancio dopo il passo lungo: peso {nome}", v, a)
controlla("lo slancio dopo il passo lungo: bias", bias, -3.55)
angoli = [(e, pg, c) for e in (0, 1) for pg in (0, 1) for c in (0, 1)]
controlla("la spinta più alta su tutte le serate possibili", max(float(slancio @ a) + bias for a in angoli), -3.55)

leaky = lambda z, alfa=0.01: max(alfa * z, z)
controlla("la scogliera con una fessura, in −3,55", leaky(-3.55), -0.0355, 0.0001)

✓  lo slancio dopo il passo lungo: peso energia: calcolato -0.04, nel libro -0.04
✓  lo slancio dopo il passo lungo: peso pioggia: calcolato -2.55, nel libro -2.55
✓  lo slancio dopo il passo lungo: peso compagno: calcolato -1.55, nel libro -1.55
✓  lo slancio dopo il passo lungo: bias: calcolato -3.55, nel libro -3.55
✓  la spinta più alta su tutte le serate possibili: calcolato -3.55, nel libro -3.55
✓  la scogliera con una fessura, in −3,55: calcolato -0.0355, nel libro -0.0355


## Quanto costa allenare un modello grande

*Nel racconto: molte vite umane in fila per una scheda sola, qualche settimana con migliaia.*

In [10]:
N, D = 7e10, 1.5e13
C = 6 * N * D
controlla("operazioni (× 10²⁴)", C / 1e24, 6.3)
secondi = C / 4e14
controlla("con una scheda: secondi (× 10¹⁰)", secondi / 1e10, 1.6, 0.05)
anni = secondi / (365.25 * 24 * 3600)
controlla("con una scheda: anni", anni, 500, 1)
controlla("vite da 80 anni", anni / 80, 6, 0.3)
controlla("con diecimila schede: secondi (× 10⁶)", secondi / 1e4 / 1e6, 1.6, 0.05)
controlla("con diecimila schede: giorni", secondi / 1e4 / 86400, 18, 0.5)

parole = D * 3 / 4
controlla("parole (× 10¹³): le undicimila miliardi del racconto", parole / 1e13, 1.1, 0.05)
al_giorno = 8 * 60 * 250
conta("parole lette al giorno", al_giorno, 120000)
controlla("parole lette in un anno (milioni)", al_giorno * 365 / 1e6, 44, 0.5)
controlla("anni per arrivare in fondo (× 10⁵), nei conti", parole / (al_giorno * 365) / 1e5, 2.6, 0.05)
controlla("anni per arrivare in fondo (migliaia), nel racconto «circa 250 mila»", 11000e9 / 44e6 / 1e3, 250)
controlla("la regola di Chinchilla per 70 miliardi di parametri (miliardi di token)", 20 * 70, 1400)

✓  operazioni (× 10²⁴): calcolato 6.3, nel libro 6.3
✓  con una scheda: secondi (× 10¹⁰): calcolato 1.575, nel libro 1.6
✓  con una scheda: anni: calcolato 499.1, nel libro 500
✓  vite da 80 anni: calcolato 6.239, nel libro 6
✓  con diecimila schede: secondi (× 10⁶): calcolato 1.575, nel libro 1.6
✓  con diecimila schede: giorni: calcolato 18.23, nel libro 18
✓  parole (× 10¹³): le undicimila miliardi del racconto: calcolato 1.125, nel libro 1.1
✓  parole lette al giorno: calcolato 120.000, nel libro 120.000
✓  parole lette in un anno (milioni): calcolato 43.8, nel libro 44
✓  anni per arrivare in fondo (× 10⁵), nei conti: calcolato 2.568, nel libro 2.6
✓  anni per arrivare in fondo (migliaia), nel racconto «circa 250 mila»: calcolato 250, nel libro 250
✓  la regola di Chinchilla per 70 miliardi di parametri (miliardi di token): calcolato 1400, nel libro 1400


## La flotta: i tre parallelismi

*Nel racconto: ogni barca scandaglia il suo tratto; l'equipaggio in cui ognuno governa la sua vela.*

In [11]:
# K barche, ciascuna con il suo mini-batch: la media dei K gradienti è il gradiente del mini-batch K volte più grande
rng = np.random.default_rng(0)
gradienti = rng.normal(size=(8, 32, 11))            # 8 schede, 32 esempi ciascuna, 11 parametri
assert np.allclose(gradienti.mean(axis=1).mean(axis=0), gradienti.reshape(-1, 11).mean(axis=0))
print("✓  la media delle medie di 8 schede è la media di un mini-batch da 256")

✓  la media delle medie di 8 schede è la media di un mini-batch da 256


## Gli ottimizzatori

*Nel racconto: l'abbrivio; ogni peso con il suo passo.*

In [12]:
beta, g, v = 0.9, 1.0, 0.0
for _ in range(200):
    v = beta * v + g
controlla("l'abbrivio con un gradiente costante: la velocità tende a (volte g)", v, 10)
v = 0.0
for k in range(200):
    v = beta * v + (g if k % 2 == 0 else -g)
print(f"   se il segno cambia a ogni passo, la velocità resta a {abs(v):.2f} g invece di 10 g: le spinte si annullano quasi del tutto")

byte = 2 + 2 + 4 + 4 + 4       # peso a 16 bit, gradiente, copia a 32 bit, due momenti
conta("byte per parametro con AdamW, in precisione mista", byte, 16)
controlla("70 miliardi di parametri, per allenare (GB)", 70e9 * byte / 1e9, 1100, 25)
controlla("per usarlo, a 16 bit (GB)", 70e9 * 2 / 1e9, 140)

✓  l'abbrivio con un gradiente costante: la velocità tende a (volte g): calcolato 10, nel libro 10
   se il segno cambia a ogni passo, la velocità resta a 0.53 g invece di 10 g: le spinte si annullano quasi del tutto
✓  byte per parametro con AdamW, in precisione mista: calcolato 16, nel libro 16
✓  70 miliardi di parametri, per allenare (GB): calcolato 1120, nel libro 1100
✓  per usarlo, a 16 bit (GB): calcolato 140, nel libro 140


## Il passo che cambia

*Nel racconto: «si comincia piano, poi si allunga, e verso la fine si accorcia di nuovo».*

In [13]:
eta_min, eta_max, T = 0.0, 1.0, 1000
coseno = lambda t: eta_min + 0.5 * (eta_max - eta_min) * (1 + math.cos(math.pi * t / T))
controlla("all'inizio il passo è il massimo", coseno(0), eta_max)
controlla("a metà strada è a metà", coseno(T / 2), (eta_max + eta_min) / 2)
controlla("alla fine è il minimo", coseno(T), eta_min)

✓  all'inizio il passo è il massimo: calcolato 1, nel libro 1
✓  a metà strada è a metà: calcolato 0.5, nel libro 0.5
✓  alla fine è il minimo: calcolato 0, nel libro 0


## Kata per casa

**Esercizio 1.** Serata (1; 0; 0), verità y = 1, pesi (2,0; −1,5; 0,1), bias −0,5, learning rate 0,5.

In [14]:
w = np.array([2.0, -1.5, 0.1]); b = -0.5; x = np.array([1, 0, 0]); y = 1; eta = 0.5
z = float(w @ x) + b
p = sigmoide(z)
controlla("spinta", z, 1.5)
controlla("p prima", p, 0.818, 0.001)
controlla("segnale d'errore", p - y, -0.182, 0.001)
w_dopo, b_dopo = w - eta * (p - y) * x, b - eta * (p - y)
controlla("w₁ dopo", w_dopo[0], 2.091, 0.001)
controlla("w₂ dopo, invariato", w_dopo[1], -1.5)
controlla("w₃ dopo, invariato", w_dopo[2], 0.1)
controlla("bias dopo", b_dopo, -0.409, 0.001)
z_dopo = float(w_dopo @ x) + b_dopo
controlla("spinta dopo", z_dopo, 1.682, 0.001)
controlla("p dopo", sigmoide(z_dopo), 0.843, 0.001)

✓  spinta: calcolato 1.5, nel libro 1.5
✓  p prima: calcolato 0.8176, nel libro 0.818
✓  segnale d'errore: calcolato -0.1824, nel libro -0.182
✓  w₁ dopo: calcolato 2.091, nel libro 2.091
✓  w₂ dopo, invariato: calcolato -1.5, nel libro -1.5
✓  w₃ dopo, invariato: calcolato 0.1, nel libro 0.1
✓  bias dopo: calcolato -0.4088, nel libro -0.409
✓  spinta dopo: calcolato 1.682, nel libro 1.682
✓  p dopo: calcolato 0.8432, nel libro 0.843
